# get similarity convergence

In [70]:
import os
import platform

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

In [71]:
os_system = platform.system() # 맥북은 Darwin, 윈도우는 Windows

# 현재 프로젝트 폴더 위치 지정. os.getcwd()는 지금 코드 실행하는 현 위치를 출력해줍니다.
research1_dir = os.getcwd()

# data/processed 폴더 위치 지정
processed_data_dir = research1_dir + ('\\data\\processed\\' if os_system == 'Windows' else '/data/processed/')

# graph 이미지 저장할 폴더 위치 지정
graph_image_dir = research1_dir + ('\\graph' if os_system == 'Windows' else '/graph')

In [72]:
# 유사도, 유사도평균 coherence값을 저장한 테이블 읽어오기
tbl_data = pd.read_csv(processed_data_dir + 'similarity_coherence_data_300.csv', index_col=0, keep_default_na=False)
tbl_data[0:3]

,similarity_abuse1_money,similarity_abuse2_money,similarity_abuse3_money,similarity_abuse4_money,similarity_abuse5_money,similarity_abuse6_money,similarity_abuse7_money,similarity_abuse8_money,similarity_abuse9_money,similarity_abuse10_money,...,coherence_mirror_friend,coherence_family_friend,coherence_abuse_relationships,coherence_tear_relationships,coherence_mirror_relationships,coherence_family_relationships,coherence_abuse_family,coherence_tear_family,coherence_mirror_family,coherence_family_family
subject,,,,,,,,,,,,,,,,,,,,,
1,,,,0.22584298,0.14024074,0.24353775,0.05485441,0.1701982,,0.28780317,...,0.112557344,0.15566784,0.09703642,0.11058131,0.03194098,0.098542534,0.14264251,0.20547445,0.12504256,0.12360635
2,0.14405176,0.14405176,0.092848584,,0.06925227,,,,0.08810924,0.17968282,...,0.16862541,0.09250904,0.056974217,0.063827634,0.068257675,0.04495881,0.07901419,0.06314863,0.15124714,0.10751439
3,0.17696357,0.15549657,-0.045841508,0.124517485,0.10273577,0.22273225,0.08812668,0.059070513,-0.025725836,0.029017754,...,0.11380485,0.12774289,0.0432437,0.07309365,0.061680965,0.06916267,0.103082016,0.13591379,0.12141059,0.14439833


In [73]:
def get_smoothed_similarity_func(similarity_df: pd.DataFrame, i_subject: int, topic: str):
    # trial 번호와 해당 trial에 대한 similarity 값을 배열로 변환
    trial_numbers = np.array(list(range(1, 41)))

    # 특정 피험자의 유사도 점수들 받아오기
    similarity_values = similarity_df.iloc[i_subject].tolist()
    similarity_values = np.array([float(value) if value != '' else 0.0 for value in similarity_values])

    # 데이터를 보간하는 함수 생성
    interpolation_function = interp1d(trial_numbers, similarity_values, kind='quadratic')

    # 정수값에 대한 데이터 추출
    integer_trial_numbers = np.arange(1, 41)
    integer_similarity_values = interpolation_function(integer_trial_numbers)

    # 부드러운 곡선을 위해 trial 번호를 더 자세히 나누기
    fine_trial_numbers = np.linspace(1, 40, 400)
    smoothed_similarity_values = interpolation_function(fine_trial_numbers)

    # 1차 함수 (선형 회귀)를 생성하여 예측값 얻기
    z = np.polyfit(integer_trial_numbers, integer_similarity_values, 1)
    p = np.poly1d(z)
    predicted_values = p(integer_trial_numbers)

    # 기울기와 절편 구하기
    slope = z[0]
    intercept = z[1]

    # 그래프 저장할 위치
    graph_image_path = (f'\\graph\\{topic}\\Subject_{i_subject}_similarity_curve.png' if os_system == 'Windows' else f'{graph_image_dir}/{topic}/Subject_{i_subject}_similarity_curve.png')
    
    # 그래프 생성
    plt.figure(figsize=(10, 5))
    plt.plot(integer_trial_numbers, predicted_values, label='Linear Regression', color='g', linestyle='-')
    plt.plot(fine_trial_numbers, smoothed_similarity_values, label='Smoothed similarities', color='r')
    plt.scatter(integer_trial_numbers, integer_similarity_values, label='Real similarity Data', marker='o', color='b')
    plt.xlabel('Trial')
    plt.ylabel('Similarity Value')
    plt.title(f'Subject {i_subject}: Smoothed Similarity Curve')
    plt.legend()
    plt.grid(True)
    plt.savefig(graph_image_path)
    # plt.show()

    # 그래프 표시하지 않음
    plt.close()

    return predicted_values, (slope, intercept)


In [74]:
def add_convergence_values_to_df(start_index: int,end_index: int, dataframe: pd.DataFrame, seed_word: str, target_word: str):
    similarity_seed_target = tbl_data.iloc[:, start_index:end_index]
    slopes = []
    intercepts = []

    for i_subject in range(len(dataframe)):
        predicted_values, (slope, intercept) = get_smoothed_similarity_func(similarity_df = similarity_seed_target,
                                                                            i_subject = i_subject,
                                                                            topic = f'{seed_word}_{target_word}')
        slopes.append(slope)
        intercepts.append(intercept)
        
    # 모든 피험자의 convergence값을 얻은 후,
    tbl_data[f'convergence_slope_{seed_word}_{target_word}'] = slopes
    tbl_data[f'convergence_intercept_{seed_word}_{target_word}'] = intercepts

### target: money

In [75]:
add_convergence_values_to_df(start_index=0,
                             end_index=40,
                             dataframe=tbl_data,
                             seed_word='abuse',
                             target_word='money')
add_convergence_values_to_df(start_index=40,
                             end_index=80,
                             dataframe=tbl_data,
                             seed_word='tear',
                             target_word='money')
add_convergence_values_to_df(start_index=80,
                             end_index=120,
                             dataframe=tbl_data,
                             seed_word='mirror',
                             target_word='money')
add_convergence_values_to_df(start_index=120,
                             end_index=160,
                             dataframe=tbl_data,
                             seed_word='family',
                             target_word='money')

### target: friend

In [90]:
add_convergence_values_to_df(start_index=160,
                             end_index=200,
                             dataframe=tbl_data,
                             seed_word='abuse',
                             target_word='friend')
add_convergence_values_to_df(start_index=200,
                             end_index=240,
                             dataframe=tbl_data,
                             seed_word='tear',
                             target_word='friend')
add_convergence_values_to_df(start_index=240,
                             end_index=280,
                             dataframe=tbl_data,
                             seed_word='mirror',
                             target_word='friend')
add_convergence_values_to_df(start_index=280,
                             end_index=320,
                             dataframe=tbl_data,
                             seed_word='family',
                             target_word='friend')

### target: relationships

In [94]:
add_convergence_values_to_df(start_index=320,
                             end_index=360,
                             dataframe=tbl_data,
                             seed_word='abuse',
                             target_word='relationships')
add_convergence_values_to_df(start_index=360,
                             end_index=400,
                             dataframe=tbl_data,
                             seed_word='tear',
                             target_word='relationships')
add_convergence_values_to_df(start_index=400,
                             end_index=440,
                             dataframe=tbl_data,
                             seed_word='mirror',
                             target_word='relationships')
add_convergence_values_to_df(start_index=440,
                             end_index=480,
                             dataframe=tbl_data,
                             seed_word='family',
                             target_word='relationships')

### target: family

In [95]:
add_convergence_values_to_df(start_index=480,
                             end_index=520,
                             dataframe=tbl_data,
                             seed_word='abuse',
                             target_word='family')
add_convergence_values_to_df(start_index=520,
                             end_index=560,
                             dataframe=tbl_data,
                             seed_word='tear',
                             target_word='family')
add_convergence_values_to_df(start_index=560,
                             end_index=600,
                             dataframe=tbl_data,
                             seed_word='mirror',
                             target_word='family')
add_convergence_values_to_df(start_index=600,
                             end_index=640,
                             dataframe=tbl_data,
                             seed_word='family',
                             target_word='family')

## save csv

In [97]:
tbl_data[0:3]

,similarity_abuse1_money,similarity_abuse2_money,similarity_abuse3_money,similarity_abuse4_money,similarity_abuse5_money,similarity_abuse6_money,similarity_abuse7_money,similarity_abuse8_money,similarity_abuse9_money,similarity_abuse10_money,...,convergence_slope_family_relationships,convergence_intercept_family_relationships,convergence_slope_abuse_family,convergence_intercept_abuse_family,convergence_slope_tear_family,convergence_intercept_tear_family,convergence_slope_mirror_family,convergence_intercept_mirror_family,convergence_slope_family_family,convergence_intercept_family_family
subject,,,,,,,,,,,,,,,,,,,,,
1,,,,0.22584298,0.14024074,0.24353775,0.05485441,0.1701982,,0.28780317,...,-0.002119,0.090253,-0.000834,0.102684,-0.000208,0.189195,0.002689,0.035539,-0.003175,0.123800
2,0.14405176,0.14405176,0.092848584,,0.06925227,,,,0.08810924,0.17968282,...,-0.000376,0.035798,-0.001524,0.082598,-0.002419,0.093794,0.000363,0.094650,-0.002894,0.126531
3,0.17696357,0.15549657,-0.045841508,0.124517485,0.10273577,0.22273225,0.08812668,0.059070513,-0.025725836,0.029017754,...,-0.000122,0.064744,-0.004125,0.177340,0.001362,0.087603,-0.001274,0.117165,-0.000408,0.138320


In [99]:
# 단어 있는 버전 csv 저장
tbl_data.to_csv(processed_data_dir + 'convergence.csv')
